In [1]:
from concurrent.futures import ProcessPoolExecutor, as_completed
from pathlib import Path

import os
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import pandas as pd
import seaborn as sns
import pickle as pkl
from tqdm import tqdm
import csv
from joblib import Parallel, delayed
from collections import Counter
from itertools import product
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_selection import SelectKBest, chi2, f_regression

In [2]:
X_byte_bigram_all_df = pd.read_csv("byte_files_bigram_df.csv")

X_byte_bigram_all_df.head(2)


,ID,00 00,00 01,00 02,00 03,00 04,00 05,00 06,00 07,00 08,...,?? f7,?? f8,?? f9,?? fa,?? fb,?? fc,?? fd,?? fe,?? ff,?? ??
0,01azqd4InC7m9JpocGv5,274425,1269,1029,1469,1227,1144,1437,1263,1174,...,0,0,0,0,0,0,0,0,0,1819
1,01IsoiSMh5gxyDYTl4CB,21075,752,73,48,175,12,10,11,42,...,0,0,0,0,0,0,0,0,0,8580


In [81]:
# 1. 讀取標籤並合併 (這時候 final_df 包含所有資料)
with open('featurization/class_labels.pkl', 'rb') as file:
    class_labels = pkl.load(file)
y_labels = class_labels.rename(columns={"Id": "ID", "Class": "Class"})
full_merged_df = pd.merge(X_byte_bigram_all_df, y_labels, on="ID", how="inner")

# 2. 讀取 ID 清單
train_ids_list = pd.read_csv("featurization/train_ids")["ID"]
test_ids_list = pd.read_csv("featurization/test_ids")["ID"]

# 3. 分別切出訓練集與測試集 (從 full_merged_df 切分)
train_df = full_merged_df[full_merged_df['ID'].isin(train_ids_list)].sort_values('ID')
test_df = full_merged_df[full_merged_df['ID'].isin(test_ids_list)].sort_values('ID')

# 4. 準備特徵 (X) 與標籤 (y)
X_train = train_df.drop(["ID", "Class"], axis=1)
y_train = train_df["Class"]
X_test = test_df.drop(["ID", "Class"], axis=1)
y_test = test_df["Class"]

# 5. 特徵選擇：Fit (只針對訓練集)
select_kbest_object = SelectKBest(score_func=chi2, k=2000)
select_kbest_object.fit(X_train, y_train)

# 6. 特徵選擇：Transform (套用到兩邊，這步最關鍵！)
# 這會把特徵數從幾萬個縮減到 2000 個
X_train_reduced = select_kbest_object.transform(X_train)
X_test_reduced = select_kbest_object.transform(X_test)

# 7. (選做) 建立分數表，方便觀察
most_imp_byte_bigram_feature_score_df = pd.DataFrame({
    'Feature': X_train.columns,
    'Score': select_kbest_object.scores_
}).sort_values(by='Score', ascending=False)

# 被選中的column name
selected_mask = select_kbest_object.get_support()
selected_feature_names = X_train.columns[selected_mask]

# 完整的train df 包含ID
X_train_reduced_df = pd.DataFrame(
    X_train_reduced, 
    columns=selected_feature_names, 
    index=X_train.index
)
X_train_reduced_df.insert(0, 'ID', train_df['ID'])

# 完整的test df 包含ID
X_test_reduced_df = pd.DataFrame(
    X_test_reduced, 
    columns=selected_feature_names, 
    index=X_test.index
)
X_test_reduced_df.insert(0, 'ID', test_df['ID'])

X_train_reduced_df.to_csv("featurization/featurization_final/top_1000_imp_byte_bigram_df_train.csv",index=False)
X_test_reduced_df.to_csv("featurization/featurization_final/top_1000_imp_byte_bigram_df_test.csv",index=False)